# Investigations to Neo4j Desktop (Local)

Este notebook carga Investigations a tu instancia local de Neo4j Desktop.

## ⚠️ Requisitos:

1. **Neo4j Desktop** ejecutándose
2. **Recalls ya subidos** (notebook 4.1.1)
3. **Archivo CSV** ya generado: `data/neo4j/exports/investigations_neo4j_ready.csv`


In [1]:
import pandas as pd
from neo4j import GraphDatabase
from pathlib import Path

print("="*70)
print("CONFIGURACION NEO4J DESKTOP LOCAL")
print("="*70)

# Credenciales de Neo4j Desktop (LOCAL)
NEO4J_URI  = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "proyectotec"  # Cambiar a tu password real

# Inicializar driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# Verificar conexión
try:
    driver.verify_connectivity()
    print(f"[OK] Conectado a Neo4j Desktop en {NEO4J_URI}")
except Exception as e:
    print(f"[ERROR] No se puede conectar: {e}")


CONFIGURACION NEO4J DESKTOP LOCAL
[OK] Conectado a Neo4j Desktop en bolt://localhost:7687


## Crear Constraints y Verificar Recalls


In [3]:
# Crear constraints necesarios (Complaint se usa para futuros datos)
CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (i:Investigation) REQUIRE i.id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Complaint) REQUIRE c.id IS UNIQUE",
]

with driver.session(database="neo4j") as s:
    for q in CONSTRAINTS:
        try:
            s.run(q)
            print(f"[OK] {q[:50]}")
        except:
            pass
    
    recalls = s.run("MATCH (r:Recall) RETURN count(r) AS n").single()['n']
    if recalls == 0:
        print("[!] No hay Recalls en Neo4j. Ejecuta primero el notebook 4.1.1")
    else:
        print(f"[OK] Recalls existentes: {recalls:,}")


[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (i:Investigati
[OK] CREATE CONSTRAINT IF NOT EXISTS FOR (c:Complaint) 
[OK] Recalls existentes: 12,760


## Cypher para Subir Investigations


In [4]:
CYPHER_UPSERT_INV = """
UNWIND $rows AS row

MERGE (i:Investigation {id: row.action_no})
  SET i.action_no  = row.action_no,
      i.make       = row.make,
      i.model      = row.model,
      i.year       = CASE WHEN row.year IS NULL OR row.year = '' THEN NULL ELSE toInteger(row.year) END,
      i.component  = row.component,
      i.open_date  = coalesce(row.open_date, ''),
      i.close_date = coalesce(row.close_date, ''),
      i.subject    = coalesce(row.subject, ''),
      i.summary    = coalesce(row.summary, ''),
      i.campaign_no = coalesce(row.campaign_no, '')

FOREACH (_ IN CASE WHEN row.make <> '' THEN [1] ELSE [] END |
  MERGE (mk:Make {name: row.make})
  MERGE (i)-[:OF_MAKE]->(mk)
)

FOREACH (_ IN CASE WHEN row.model <> '' AND row.make <> '' THEN [1] ELSE [] END |
  MERGE (md:Model {name: row.model, make: row.make})
  MERGE (i)-[:OF_MODEL]->(md)
)

MERGE (c1:Component {name: row.comp_l1})
  ON CREATE SET c1.name_lower = toLower(row.comp_l1)
  ON MATCH  SET c1.name_lower = coalesce(c1.name_lower, toLower(row.comp_l1))

MERGE (c2:Component {name: CASE WHEN row.comp_l2 <> '' THEN row.comp_l2 ELSE row.comp_l1 END})
  ON CREATE SET c2.name_lower = toLower(CASE WHEN row.comp_l2 <> '' THEN row.comp_l2 ELSE row.comp_l1 END)
FOREACH (_ IN CASE WHEN row.comp_l2 <> '' THEN [1] ELSE [] END |
  MERGE (c1)<-[:SUB_OF]-(c2)
)

MERGE (c3:Component {name: CASE WHEN row.comp_l3 <> '' THEN row.comp_l3 ELSE c2.name END})
  ON CREATE SET c3.name_lower = toLower(CASE WHEN row.comp_l3 <> '' THEN row.comp_l3 ELSE c2.name END)
FOREACH (_ IN CASE WHEN row.comp_l3 <> '' THEN [1] ELSE [] END |
  MERGE (c2)<-[:SUB_OF]-(c3)
)

MERGE (c4:Component {name: CASE WHEN row.comp_l4 <> '' THEN row.comp_l4 ELSE c3.name END})
  ON CREATE SET c4.name_lower = toLower(CASE WHEN row.comp_l4 <> '' THEN row.comp_l4 ELSE c3.name END)
FOREACH (_ IN CASE WHEN row.comp_l4 <> '' THEN [1] ELSE [] END |
  MERGE (c3)<-[:SUB_OF]-(c4)
)

MERGE (c5:Component {name: CASE WHEN row.comp_l5 <> '' THEN row.comp_l5 ELSE c4.name END})
  ON CREATE SET c5.name_lower = toLower(CASE WHEN row.comp_l5 <> '' THEN row.comp_l5 ELSE c4.name END)
FOREACH (_ IN CASE WHEN row.comp_l5 <> '' THEN [1] ELSE [] END |
  MERGE (c4)<-[:SUB_OF]-(c5)
)

WITH row, i, c1, c2, c3, c4, c5
WITH i,
     CASE
       WHEN row.comp_l5 <> '' THEN c5
       WHEN row.comp_l4 <> '' THEN c4
       WHEN row.comp_l3 <> '' THEN c3
       WHEN row.comp_l2 <> '' THEN c2
       ELSE c1
     END AS leaf,
     row
MERGE (i)-[:MENTIONS]->(leaf)

WITH row, i
WHERE row.campaign_no <> ''
OPTIONAL MATCH (r:Recall {id: row.campaign_no})
WITH i, r
WHERE r IS NOT NULL
MERGE (i)-[:RELATES_TO]->(r);
"""


## Cargar CSV y Subir


In [5]:
CSV = Path("../data/neo4j/exports/investigations_neo4j_ready.csv")
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print(f"[i] CSV cargado: {len(df):,} filas")
print(f"[i] Columnas: {list(df.columns)}")
print(f"[i] Filas con camp_no: {(df['campaign_no'] != '').sum():,}")


[i] CSV cargado: 152,193 filas
[i] Columnas: ['action_no', 'make', 'model', 'year', 'component', 'open_date', 'close_date', 'subject', 'summary', 'campaign_no', 'comp_l1', 'comp_l2', 'comp_l3', 'comp_l4', 'comp_l5']
[i] Filas con camp_no: 125,122


In [6]:
CSV = Path("../data/neo4j/exports/investigations_neo4j_ready.csv")
df = pd.read_csv(CSV, dtype=str, keep_default_na=False)
print(f"[i] CSV cargado: {len(df):,} filas")
print(f"[i] Filas con camp_no: {(df['campaign_no'] != '').sum():,}")

def ingest(csv_df, cypher, batch=1000):
    total, i = len(csv_df), 0
    with driver.session(database="neo4j") as s:
        while i < total:
            rows = csv_df.iloc[i:i+batch].to_dict('records')
            s.run(cypher, rows=rows)
            i += batch
            if i % 5000 == 0 or i >= total:
                print(f"→ {min(i,total)}/{total}")
    print("Ingesta completa ✅")

print("\n" + "="*70)
print("INGESTA COMPLETA DE INVESTIGATIONS")
print("="*70)
ingest(df, CYPHER_UPSERT_INV, batch=1000)

with driver.session(database="neo4j") as s:
    invs = s.run("MATCH (i:Investigation) RETURN count(i) AS n").single()['n']
    rels_to_recalls = s.run("MATCH (i:Investigation)-[:RELATES_TO]->(r:Recall) RETURN count(i) AS n").single()['n']
    
    print(f"\n[OK] Ingesta completada!")
    print(f"   Investigations: {invs:,}")
    print(f"   Vinculadas con Recalls: {rels_to_recalls:,}")


[i] CSV cargado: 152,193 filas
[i] Filas con camp_no: 125,122

INGESTA COMPLETA DE INVESTIGATIONS
→ 5000/152193
→ 10000/152193
→ 15000/152193
→ 20000/152193
→ 25000/152193
→ 30000/152193
→ 35000/152193
→ 40000/152193
→ 45000/152193
→ 50000/152193
→ 55000/152193
→ 60000/152193
→ 65000/152193
→ 70000/152193
→ 75000/152193
→ 80000/152193
→ 85000/152193
→ 90000/152193
→ 95000/152193
→ 100000/152193
→ 105000/152193
→ 110000/152193
→ 115000/152193
→ 120000/152193
→ 125000/152193
→ 130000/152193
→ 135000/152193
→ 140000/152193
→ 145000/152193
→ 150000/152193
→ 152193/152193
Ingesta completa ✅

[OK] Ingesta completada!
   Investigations: 4,031
   Vinculadas con Recalls: 737
